# 05 — Predictive Modeling Notebook

**Owner:** M5 — Thư  
**Purpose:** run/review the M5 model pipeline and prepare the PSM propensity-score handoff for M6.

This notebook is organized into **2 layers**:

```text
LAYER 1 — Core M5 Modeling
Run/review churn model, calibration, value model, expected profit, and high-risk candidate outputs.

LAYER 2 — PSM Support for M6
Run/review the Propensity Score model after M4 provides treatment flags.
```

Core reusable logic stays in scripts:

```text
scripts/modeling.py               -> full M5 churn/value/profit pipeline
scripts/psm_propensity_score.py   -> PSM propensity-score support for M6
```

## How to use this notebook

### Recommended order

1. Run **Setup**.
2. Run **Layer 1** if you need to regenerate or review M5 outputs.
3. Run **Layer 2** only after M4 provides the real PSM treatment flag file.

### Important switches

The actual feature sources are controlled by `config/paths.yaml`:

```text
models/final_ML_features_linear.csv -> Logistic Regression, PSM propensity
models/final_ML_features_tree.csv   -> Random Forest, Extra Trees, XGBoost, tree value models
```

Notebook run switches below only decide whether to execute scripts from the notebook. They do not change model logic.

## 0. Setup paths and imports

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'config').exists() and (PROJECT_ROOT.parent / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = (PROJECT_ROOT / 'config' / 'paths.yaml').resolve()
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = MODELS_DIR / 'reports'
M6_HANDOFF_DIR = MODELS_DIR / 'm6_handoff'
DIAGNOSTICS_DIR = MODELS_DIR / 'diagnostics'
ARTIFACTS_DIR = MODELS_DIR / 'artifacts'
PSM_INPUTS_DIR = MODELS_DIR / 'psm_inputs'

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

print('Project root:', PROJECT_ROOT)
print('Config:', CONFIG_PATH)
print('Models:', MODELS_DIR)
print('Linear features:', CONFIG.get('inputs', {}).get('feature_table_linear_csv'))
print('Tree features:', CONFIG.get('inputs', {}).get('feature_table_tree_csv'))

Project root: C:\Users\Thuww\DDM-Churn-Project
Config: C:\Users\Thuww\DDM-Churn-Project\config\paths.yaml
Models: C:\Users\Thuww\DDM-Churn-Project\models
Linear features: models/final_ML_features_linear.csv
Tree features: models/final_ML_features_tree.csv


# LAYER 1 — Core M5 Modeling Pipeline

This layer is the main M5 model workflow. It answers:

```text
1. Which households are likely to churn?
2. Are churn probabilities calibrated enough for business formulas?
3. If a household is retained/active, how much short-term value can it generate?
4. Under business assumptions, does treatment have positive expected profit?
5. Which customers should be passed to M6 as high-risk candidates?
```

This layer is handled by:

```bash
python scripts/modeling.py --config config/paths.yaml
```

## 1.1 Validate core M5 inputs

M4 now hands off **two household-level feature tables**:

```text
models/final_ML_features_linear.csv
models/final_ML_features_tree.csv
```

M5 must keep one synchronized split, but route each model family to the correct feature source:

| Feature table | Used by |
|---|---|
| `final_ML_features_linear.csv` | Logistic Regression, Ridge, PSM propensity model |
| `final_ML_features_tree.csv` | Random Forest, Extra Trees, XGBoost, tree value models |

Both tables must contain the same `household_key` and the same `churn_flag`.


In [2]:
inputs_cfg = CONFIG.get('inputs', {})
id_col = CONFIG.get('modeling', {}).get('id_col', 'household_key')
target_col = CONFIG.get('modeling', {}).get('target_col', 'churn_flag')

linear_path = PROJECT_ROOT / inputs_cfg.get('feature_table_linear_csv', inputs_cfg.get('feature_table_csv'))
tree_path = PROJECT_ROOT / inputs_cfg.get('feature_table_tree_csv', inputs_cfg.get('feature_table_csv'))

print('Linear feature table:', linear_path.relative_to(PROJECT_ROOT), linear_path.exists())
print('Tree feature table:', tree_path.relative_to(PROJECT_ROOT), tree_path.exists())

linear_features = pd.read_csv(linear_path)
tree_features = pd.read_csv(tree_path)

checks = {
    'linear_shape': linear_features.shape,
    'tree_shape': tree_features.shape,
    'linear_duplicate_ids': int(linear_features[id_col].duplicated().sum()),
    'tree_duplicate_ids': int(tree_features[id_col].duplicated().sum()),
    'linear_missing_values': int(linear_features.isna().sum().sum()),
    'tree_missing_values': int(tree_features.isna().sum().sum()),
    'same_household_set': set(linear_features[id_col]) == set(tree_features[id_col]),
    'same_churn_rate': float(linear_features[target_col].mean()) == float(tree_features[target_col].mean()),
    'linear_churn_rate': float(linear_features[target_col].mean()),
    'tree_churn_rate': float(tree_features[target_col].mean()),
    'linear_only_columns': sorted(set(linear_features.columns) - set(tree_features.columns)),
    'tree_only_columns': sorted(set(tree_features.columns) - set(linear_features.columns)),
}
checks

Linear feature table: models\final_ML_features_linear.csv True
Tree feature table: models\final_ML_features_tree.csv True


{'linear_shape': (2499, 24),
 'tree_shape': (2499, 27),
 'linear_duplicate_ids': 0,
 'tree_duplicate_ids': 0,
 'linear_missing_values': 0,
 'tree_missing_values': 0,
 'same_household_set': True,
 'same_churn_rate': True,
 'linear_churn_rate': 0.1208483393357343,
 'tree_churn_rate': 0.1208483393357343,
 'linear_only_columns': [],
 'tree_only_columns': ['IPT_mean', 'IPT_std', 'Total_Campaigns_Received']}

## 1.2 Run the full core M5 pipeline

This cell explicitly runs:

```bash
python scripts/modeling.py --config config/paths.yaml
```

The pipeline now reads both feature tables from config and routes model families automatically:

```text
linear source -> Logistic / Ridge/  PSM-style linear models 
tree source   -> Random Forest / Extra Trees / XGBoost / tree value models
```

It regenerates the core M5 outputs:

```text
models/reports/       -> model metrics, calibration, top-K, profit summaries
models/m6_handoff/    -> high-risk / priority customer files for M6
models/diagnostics/   -> calibration, SHAP, seasonality, audit files 
models/artifacts/     -> model artifact package
```

Leave `RUN_FULL_M5_PIPELINE = False` if you only want to review existing outputs.


In [14]:
RUN_FULL_M5_PIPELINE = False

if RUN_FULL_M5_PIPELINE:
    import subprocess
    cmd = [
        sys.executable,
        str((PROJECT_ROOT / 'scripts' / 'modeling.py').resolve()),
        '--config',
        str(CONFIG_PATH),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)

summary_path = PROJECT_ROOT / 'reports' / 'internal_briefs' / 'm5_pipeline_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('Loaded pipeline summary from:', summary_path.relative_to(PROJECT_ROOT))
else:
    summary = {}
    print('No pipeline summary found yet. Set RUN_FULL_M5_PIPELINE=True to run M5.')

summary

Loaded pipeline summary from: reports\internal_briefs\m5_pipeline_summary.json


{'version': 'v3_discounted_two_part_value_shap_seasonality',
 'project_root': 'C:\\Users\\Thuww\\DDM-Churn-Project',
 'feature_rows': 2499,
 'churn_rate': 0.1208483393357343,
 'champion_churn_model': 'Logistic Regression balanced',
 'champion_churn_feature_source': 'linear',
 'calibration_method': 'isotonic',
 'champion_threshold': 0.09,
 'test_PR_AUC_calibrated': 0.31741537446065193,
 'test_F2_score_calibrated': 0.47761194029850745,
 'test_brier_score_calibrated': 0.09331076804905715,
 'active_model': 'Active Random Forest__isotonic',
 'active_model_feature_source': 'tree',
 'value_model': 'XGBoost Regressor',
 'value_model_feature_source': 'tree',
 'linear_feature_columns': 22,
 'tree_feature_columns': 25,
 'annual_discount_rate': 0.08,
 'base_profitable_customers': 0,
 'shap_status_file': 'C:\\Users\\Thuww\\DDM-Churn-Project\\reports\\internal_briefs\\M5_shap_status.json',
 'outputs_dir': 'C:\\Users\\Thuww\\DDM-Churn-Project\\models',
 'reports_outputs_dir': 'C:\\Users\\Thuww\\DDM-C

## 1.3 Load core M5 output tables

These helper functions read outputs from the organized M5 folders. They also fall back to `models/` root for older runs.

In [4]:
def read_m5_output(filename: str, folder: Path):
    candidates = [folder / filename, MODELS_DIR / filename]
    for path in candidates:
        if path.exists():
            print(f'Loaded: {path.relative_to(PROJECT_ROOT)}')
            return pd.read_csv(path)
    print(f'Missing: {filename}')
    return pd.DataFrame()

def show_selected(df: pd.DataFrame, columns: list[str], n: int = 10):
    if df.empty:
        return df
    available = [c for c in columns if c in df.columns]
    return df[available].head(n)

model_metrics = read_m5_output('model_metrics.csv', REPORTS_DIR)
calibration_summary = read_m5_output('calibration_summary.csv', REPORTS_DIR)
champion_test_metrics = read_m5_output('champion_test_metrics.csv', REPORTS_DIR)
active_model_metrics = read_m5_output('active_model_metrics.csv', DIAGNOSTICS_DIR)
value_model_metrics = read_m5_output('value_model_metrics.csv', REPORTS_DIR)
top_k_precision = read_m5_output('top_k_precision_summary.csv', REPORTS_DIR)
ranking_deciles = read_m5_output('ranking_decile_performance.csv', REPORTS_DIR)
scenario_profit = read_m5_output('scenario_profit_summary.csv', REPORTS_DIR)
profit_threshold = read_m5_output('profit_threshold_analysis.csv', REPORTS_DIR)


Loaded: models\reports\model_metrics.csv
Loaded: models\reports\calibration_summary.csv
Loaded: models\reports\champion_test_metrics.csv
Loaded: models\diagnostics\active_model_metrics.csv
Loaded: models\reports\value_model_metrics.csv
Loaded: models\reports\top_k_precision_summary.csv
Loaded: models\reports\ranking_decile_performance.csv
Loaded: models\reports\scenario_profit_summary.csv
Loaded: models\reports\profit_threshold_analysis.csv


## 1.4 Churn model benchmark

File:

```text
models/reports/model_metrics.csv
```

Purpose:

> Compare candidate churn classifiers before final business interpretation.

Key interpretation:

- `PR-AUC`: main ranking metric for imbalanced churn.
- `F2_score`: favors recall over precision.
- `precision`: among predicted churn-risk households, how many churned.
- `recall`: among actual churners, how many were caught.
- `Brier score`: probability quality.

Because churn rate is low, global precision can look low. For M6, **top-K precision and risk ranking** matter more than global precision.

## 1.5 Calibration summary

File:

```text
models/reports/calibration_summary.csv
```

Purpose:

> Compare raw, sigmoid, and isotonic probabilities for the selected churn model.

Why it matters:

M5 uses churn probability directly in expected-profit formulas. Therefore, raw model probability should not be used if it is miscalibrated.

Interpretation:

- `raw_uncalibrated`: original model probability.
- `sigmoid`: smoother calibrated probability.
- `isotonic`: flexible calibration, but may create flat spots.
- `selected_for_champion = True`: final calibrated probability used as `p_churn_calibrated`.

In [ ]:
model_metrics_clean = model_metrics[~model_metrics['model'].astype(str).str.contains('RFECV', na=False)].copy()
show_selected(
    model_metrics_clean,
    [
        'model', 'tuned', 'features_used', 'best_cv_PR_AUC', 'best_cv_PR_AUC_std',
        'val_PR_AUC', 'val_ROC_AUC', 'val_F2_score', 'val_precision', 'val_recall',
        'test_PR_AUC', 'test_F2_score', 'test_precision', 'test_recall', 'test_brier_score'
    ],
    n=12,
)

In [6]:
show_selected(
    calibration_summary,
    [
        'champion_model', 'calibration_method', 'val_PR_AUC', 'val_ROC_AUC',
        'val_F2_score', 'val_precision', 'val_recall', 'val_brier_score',
        'val_threshold', 'test_PR_AUC', 'test_F2_score', 'test_precision', 'test_recall',
        'test_brier_score', 'test_mean_predicted_probability', 'test_actual_positive_rate',
        'test_calibration_gap_mean_minus_actual', 'val_brier_gap_from_best',
        'calibration_selection_eligible', 'selected_for_champion'
    ],
    n=10,
)

,champion_model,calibration_method,val_PR_AUC,val_ROC_AUC,val_F2_score,val_precision,val_recall,val_brier_score,val_threshold,test_PR_AUC,test_F2_score,test_precision,test_recall,test_brier_score,test_mean_predicted_probability,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,val_brier_gap_from_best,calibration_selection_eligible,selected_for_champion
0,Logistic Regression balanced,isotonic,0.416606,0.801393,0.563380,0.360360,0.655738,0.084514,0.09,0.317415,0.477612,0.336842,0.533333,0.093311,0.114873,0.12,-0.005127,0.000000,True,True
1,Logistic Regression balanced,sigmoid,0.413643,0.778184,0.543478,0.322581,0.655738,0.092589,0.14,0.347976,0.476879,0.311321,0.550000,0.093410,0.119607,0.12,-0.000393,0.008075,True,False
2,Logistic Regression balanced,raw_uncalibrated,0.413643,0.778184,0.554017,0.341880,0.655738,0.210126,0.55,0.347976,0.470588,0.320000,0.533333,0.210382,0.463430,0.12,0.343430,0.125612,False,False


## 1.6 Active model and value model

Files:

```text
models/diagnostics/active_model_metrics.csv
models/reports/value_model_metrics.csv
```

These are part of the **value layer**, not the main churn model.

### Active model

Estimates:

```text
p_future_active = P(household generates revenue in the 60-day prediction window)
```

The active model now records `feature_source`, so Logistic uses the linear table and Random Forest uses the tree table.

### Value model

Estimates:

```text
predicted_discounted_value_60d_if_active
```

Ridge uses the linear feature table. Random Forest/XGBoost regressors use the tree feature table. This is **discounted 60-day value**, not full lifetime CLV.

In [7]:
print('Active model metrics')
display(show_selected(
    active_model_metrics,
    [
        'active_model', 'calibration_method', 'feature_source', 'val_PR_AUC', 'val_brier_score',
        'val_threshold', 'test_brier_score', 'test_mean_predicted_probability',
        'test_actual_positive_rate', 'test_calibration_gap_mean_minus_actual',
        'test_TN', 'test_FP', 'test_FN', 'test_TP'
    ],
    n=10,
))

print('Value model metrics')
display(show_selected(
    value_model_metrics,
    [
        'value_model', 'feature_source', 'target', 'val_RMSE_log', 'val_MAE_log', 'val_R2_log',
        'val_MAE_revenue', 'test_RMSE_log', 'test_MAE_log', 'test_R2_log',
        'test_MAE_revenue'
    ],
    n=10,
))

Active model metrics


,active_model,calibration_method,feature_source,val_PR_AUC,val_brier_score,val_threshold,test_brier_score,test_mean_predicted_probability,test_actual_positive_rate,test_calibration_gap_mean_minus_actual,test_TN,test_FP,test_FN,test_TP
0,Active Random Forest,isotonic,tree,0.980696,0.075009,0.01,0.069763,0.889360,0.898,-0.008640,0,51,0,449
1,Active Logistic Regression,isotonic,linear,0.974158,0.076966,0.01,0.070315,0.884152,0.898,-0.013848,3,48,1,448
2,Active Random Forest,raw_uncalibrated,tree,0.982136,0.080912,0.41,0.068182,0.895866,0.898,-0.002134,0,51,0,449
3,Active Logistic Regression,raw_uncalibrated,linear,0.974106,0.082798,0.27,0.071136,0.894049,0.898,-0.003951,3,48,1,448
4,Active Random Forest,sigmoid,tree,0.982136,0.086334,0.44,0.073492,0.884791,0.898,-0.013209,0,51,0,449
5,Active Logistic Regression,sigmoid,linear,0.974106,0.086983,0.38,0.075322,0.883265,0.898,-0.014735,3,48,1,448


Value model metrics


,value_model,feature_source,target,val_RMSE_log,val_MAE_log,val_R2_log,val_MAE_revenue,test_RMSE_log,test_MAE_log,test_R2_log,test_MAE_revenue
0,XGBoost Regressor,tree,log1p(discounted_future_revenue_60d | active),0.864135,0.637375,0.536758,137.213128,0.879033,0.641718,0.526824,152.047466
1,Random Forest Regressor,tree,log1p(discounted_future_revenue_60d | active),0.869776,0.637162,0.530690,136.903366,0.894711,0.650315,0.509795,150.022317
2,Ridge Regression,linear,log1p(discounted_future_revenue_60d | active),0.921904,0.698474,0.472750,168.327220,0.946471,0.694094,0.451436,192.807768


## 1.7 Top-K ranking and profit scenarios

Files:

```text
models/reports/top_k_precision_summary.csv
models/reports/scenario_profit_summary.csv
models/reports/profit_threshold_analysis.csv
```

Purpose:

> Show whether M5 is useful for risk ranking and whether treatment looks profitable under business assumptions.

Expected-profit formula:

```text
expected_profit = p_churn_calibrated × save_rate × value_if_active × gross_margin - treatment_cost
```

If base scenario has zero profitable customers, the correct conclusion is:

> Do not auto-rollout paid vouchers. Use M5 output as candidate ranking for M6 / PSM / future campaign validation.

In [8]:
print('Top-K precision')
display(top_k_precision.head(20))

print('Scenario profit summary')
display(scenario_profit.head(20))

print('Profit threshold analysis')
display(profit_threshold.head(20))

Top-K precision


,score_name,top_k_share,top_k_customer_count,baseline_churn_rate,precision_at_k,lift_vs_baseline,churn_count_at_k,recall_at_k,min_score_in_top_k,mean_score_in_top_k,max_score_in_top_k
0,p_churn_calibrated,0.05,125,0.120848,0.504,4.170517,63,0.208609,0.370370,0.593909,1.000000
1,p_churn_calibrated,0.10,250,0.120848,0.440,3.640927,110,0.364238,0.357143,0.479380,1.000000
2,p_churn_calibrated,0.20,500,0.120848,0.332,2.747245,166,0.549669,0.086957,0.352172,1.000000
3,risk_value_score,0.05,125,0.120848,0.168,1.390172,21,0.069536,66.950510,107.033090,324.698661
4,risk_value_score,0.10,250,0.120848,0.168,1.390172,42,0.139073,48.683455,82.137208,324.698661
5,risk_value_score,0.20,500,0.120848,0.158,1.307424,79,0.261589,30.284028,59.832680,324.698661
6,expected_incremental_profit_base,0.05,125,0.120848,0.168,1.390172,21,0.069536,-4.163119,-3.662086,-0.941267
7,expected_incremental_profit_base,0.10,250,0.120848,0.168,1.390172,42,0.139073,-4.391457,-3.973285,-0.941267
8,expected_incremental_profit_base,0.20,500,0.120848,0.158,1.307424,79,0.261589,-4.621450,-4.252091,-0.941267


Scenario profit summary


,scenario,scenario_type,gross_margin,save_rate_given_treatment,treatment_cost,profitable_customer_count,profitable_customer_share,total_expected_incremental_profit_if_target_positive_only,top_30pct_risk_customer_count,total_expected_incremental_profit_if_target_top_30pct_risk
0,conservative,named,0.20,0.03,5.0,0,0.000000,0.000000,750,-3593.335497
1,base,named,0.25,0.05,5.0,0,0.000000,0.000000,750,-3423.615619
2,optimistic,named,0.30,0.08,3.0,24,0.009604,43.877914,750,-1623.341989


Profit threshold analysis


,scenario,expected_profit_column,selection_rule,selected_customer_count,selected_customer_share,baseline_churn_rate,selected_churn_rate,lift_vs_baseline,total_expected_incremental_profit,mean_expected_incremental_profit,min_expected_incremental_profit,max_expected_incremental_profit,min_p_churn_calibrated,mean_p_churn_calibrated,max_p_churn_calibrated
0,conservative,expected_incremental_profit_conservative,profit_positive,0,0.000000,0.120848,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
1,conservative,expected_incremental_profit_conservative,top_5pct_by_expected_profit,125,0.050020,0.120848,0.168000,1.390172,-544.725182,-4.357801,-4.598297,-3.051808,0.052239,0.241150,1.00
2,conservative,expected_incremental_profit_conservative,top_10pct_by_expected_profit,250,0.100040,0.120848,0.168000,1.390172,-1126.794188,-4.507177,-4.707899,-3.051808,0.052239,0.212156,1.00
3,conservative,expected_incremental_profit_conservative,top_20pct_by_expected_profit,500,0.200080,0.120848,0.158000,1.307424,-2320.501960,-4.641004,-4.818296,-3.051808,0.025316,0.187131,1.00
4,base,expected_incremental_profit_base,profit_positive,0,0.000000,0.120848,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN
5,base,expected_incremental_profit_base,top_5pct_by_expected_profit,125,0.050020,0.120848,0.168000,1.390172,-457.760797,-3.662086,-4.163119,-0.941267,0.052239,0.241150,1.00
6,base,expected_incremental_profit_base,top_10pct_by_expected_profit,250,0.100040,0.120848,0.168000,1.390172,-993.321225,-3.973285,-4.391457,-0.941267,0.052239,0.212156,1.00
7,base,expected_incremental_profit_base,top_20pct_by_expected_profit,500,0.200080,0.120848,0.158000,1.307424,-2126.045749,-4.252091,-4.621450,-0.941267,0.025316,0.187131,1.00
8,optimistic,expected_incremental_profit_optimistic,profit_positive,24,0.009604,0.120848,0.083333,0.689570,43.877914,1.828246,0.003553,4.792768,0.235294,0.289945,0.75
9,optimistic,expected_incremental_profit_optimistic,top_5pct_by_expected_profit,125,0.050020,0.120848,0.168000,1.390172,-53.900730,-0.431206,-1.393188,4.792768,0.052239,0.241150,1.00


## 1.8 Core handoff files for M6

Folder:

```text
models/m6_handoff/
```

Important files:

| File | Purpose |
|---|---|
| `high_risk_customers_for_ab_test.csv` | Top-risk candidate pool. Can also support PSM subgroup analysis. |
| `priority_customers_all.csv` | Full ranking with risk/value/profit fields. |
| `churn_predictions.csv` | Full customer-level scoring output. |

These files identify high-risk households. They do **not** prove coupon treatment works.

In [9]:
high_risk = read_m5_output('high_risk_customers_for_ab_test.csv', M6_HANDOFF_DIR)
priority_all = read_m5_output('priority_customers_all.csv', M6_HANDOFF_DIR)

print('High-risk customer file shape:', high_risk.shape)
display(show_selected(
    high_risk,
    [
        'household_key', 'p_churn_calibrated', 'risk_rank', 'risk_decile',
        'risk_value_score', 'risk_value_rank', 'priority_segment',
        'predicted_discounted_value_60d_if_active',
        'expected_incremental_profit_base', 'recommended_treatment_action_base'
    ],
    n=10,
))

Loaded: models\m6_handoff\high_risk_customers_for_ab_test.csv
Loaded: models\m6_handoff\priority_customers_all.csv
High-risk customer file shape: (750, 65)


,household_key,p_churn_calibrated,risk_rank,risk_decile,risk_value_score,risk_value_rank,priority_segment,predicted_discounted_value_60d_if_active,expected_incremental_profit_base,recommended_treatment_action_base
0,379,1.00,1,1,40.030106,320,High Risk - Low Value,40.030106,-4.499624,A/B test candidate only; not profitable under ...
1,1876,1.00,2,1,58.988102,170,High Risk - Low Value,58.988102,-4.262649,A/B test candidate only; not profitable under ...
2,2051,1.00,3,1,68.025406,117,High Risk - Low Value,68.025406,-4.149682,A/B test candidate only; not profitable under ...
3,2315,1.00,4,1,67.584358,119,High Risk - Low Value,67.584360,-4.155196,A/B test candidate only; not profitable under ...
4,2443,1.00,5,1,75.931618,83,High Risk - Low Value,75.931620,-4.050855,A/B test candidate only; not profitable under ...
5,11,0.75,6,1,8.933791,1521,High Risk - Low Value,11.911721,-4.888328,A/B test candidate only; not profitable under ...
6,313,0.75,7,1,15.140695,1081,High Risk - Low Value,20.187593,-4.810741,A/B test candidate only; not profitable under ...
7,494,0.75,8,1,31.780512,475,High Risk - Low Value,42.374016,-4.602744,A/B test candidate only; not profitable under ...
8,682,0.75,9,1,57.736204,180,High Risk - Low Value,76.981606,-4.278297,A/B test candidate only; not profitable under ...
9,780,0.75,10,1,27.351823,575,High Risk - Low Value,36.469097,-4.658102,A/B test candidate only; not profitable under ...


# LAYER 2 — PSM Support for M6

This layer is separate from the core M5 churn model.

M6 has chosen **Propensity Score Matching (PSM)**. M5's support is only to generate:

```text
propensity_score = P(is_treated = 1 | observed covariates)
```

M6 will use this score for:

```text
matching treated/control households
balance plots
effect estimation
ROI / ROMI calculation
```

M5 does **not** perform final matching or ROI calculation in this notebook.

## 2.1 PSM input contract with M4


In [10]:
psm_flag_path = PSM_INPUTS_DIR / 'psm_treatment_flags.csv'
psm_template_path = PSM_INPUTS_DIR / 'psm_treatment_flags_TEMPLATE.csv'
psm_output_path = M6_HANDOFF_DIR / 'propensity_scores_for_psm.csv'

print('Expected PSM input from M4:', psm_flag_path.relative_to(PROJECT_ROOT))
print('PSM output for M6:', psm_output_path.relative_to(PROJECT_ROOT))

if psm_flag_path.exists():
    flags_preview = pd.read_csv(psm_flag_path)
    print('Treatment flag file found:', flags_preview.shape)
    display(flags_preview.head())
else:
    print('Treatment flag file not found yet.')
    print('Ask M4 to create:', psm_flag_path.relative_to(PROJECT_ROOT))
    if psm_template_path.exists():
        print('Template exists:', psm_template_path.relative_to(PROJECT_ROOT))

Expected PSM input from M4: models\psm_inputs\psm_treatment_flags.csv
PSM output for M6: models\m6_handoff\propensity_scores_for_psm.csv
Treatment flag file found: (2499, 4)


,household_key,is_treated,treatment_source,treatment_cutoff_day
0,1,1,Heavy_Promo_User,651
1,10,0,Light/No_Promo_User,651
2,100,1,Heavy_Promo_User,651
3,1000,0,Light/No_Promo_User,651
4,1001,0,Light/No_Promo_User,651


## 2.2 Run the PSM propensity-score model

This step runs:

```bash
python scripts/psm_propensity_score.py --config config/paths.yaml
```

It fits a simple Logistic Regression model:

```text
propensity_score = P(is_treated = 1 | observed covariates)
```

The PSM script now reads the configured linear feature table:

```text
inputs.psm_feature_table_csv: models/final_ML_features_linear.csv
```

The script excludes unsafe columns such as outcome/future/profit/coupon-like variables to reduce leakage/collider-bias risk.

Set `RUN_PSM_PROPENSITY = True` only after M4 treatment flags are final.

In [11]:
RUN_PSM_PROPENSITY = True
ALLOW_DUMMY_PSM = False

if RUN_PSM_PROPENSITY:
    import subprocess
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / 'scripts' / 'psm_propensity_score.py'),
        '--config', str(CONFIG_PATH),
    ]
    if ALLOW_DUMMY_PSM:
        cmd.append('--allow-dummy')
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=PROJECT_ROOT)
else:
    print('Skipped PSM propensity score generation. Set RUN_PSM_PROPENSITY=True after M4 provides real treatment flags.')

Running: c:\Users\Thuww\AppData\Local\Programs\Python\Python312\python.exe C:\Users\Thuww\DDM-Churn-Project\scripts\psm_propensity_score.py --config C:\Users\Thuww\DDM-Churn-Project\config\paths.yaml


## 2.4 Review PSM output for M6

Final handoff file:

```text
models/m6_handoff/propensity_scores_for_psm.csv
```

Minimum columns:

| Column | Meaning |
|---|---|
| `household_key` | Household identifier |
| `is_treated` | Treatment flag from M4 |
| `propensity_score` | Estimated probability of being treated |
| `propensity_logit` | Logit transform, useful for caliper matching |
| `common_support_flag` | Whether household falls inside treated/control common support |

M6 uses this file for matching and balance plots.

In [12]:
psm_scores = read_m5_output('propensity_scores_for_psm.csv', M6_HANDOFF_DIR)
if not psm_scores.empty:
    display(show_selected(
        psm_scores,
        [
            'household_key', 'is_treated', 'propensity_score', 'propensity_logit',
            'common_support_flag', 'p_churn_calibrated', 'risk_rank', 'risk_decile',
            'priority_segment', 'predicted_discounted_value_60d_if_active'
        ],
        n=10,
    ))
else:
    print('No PSM score output yet. Run the PSM script after M4 provides real treatment flags.')

Loaded: models\m6_handoff\propensity_scores_for_psm.csv


,household_key,is_treated,propensity_score,propensity_logit,common_support_flag,p_churn_calibrated,risk_rank,risk_decile,priority_segment,predicted_discounted_value_60d_if_active
0,1,1,0.649919,0.618681,True,0.025316,2041,9,Low Risk - High Value,442.964300
1,10,0,0.152544,-1.714786,True,0.370370,90,1,High Risk - Low Value,58.968370
2,100,1,0.575973,0.306265,True,0.052239,1334,6,Low Risk - Low Value,195.466860
3,1000,0,0.361700,-0.567995,True,0.075269,919,4,Low Risk - Low Value,267.832520
4,1001,0,0.372744,-0.520461,True,0.052239,1586,7,Low Risk - High Value,355.511020
5,1002,0,0.311403,-0.793569,True,0.000000,2441,10,Low Risk - Low Value,108.107140
6,1003,0,0.434651,-0.262900,True,0.235294,352,2,High Risk - Low Value,190.017700
7,1004,0,0.225655,-1.233011,True,0.075269,920,4,Low Risk - Low Value,260.663300
8,1005,0,0.222120,-1.253353,True,0.235294,353,2,High Risk - Low Value,198.736970
9,1006,1,0.131207,-1.890328,True,0.052239,1587,7,Low Risk - Low Value,41.303062
